# Eval Overview - Uncertainty Calibration

The uncertainty calibration evaluations are a set of benchmarks that evaluate a unique output from BioJEPA-AC, the *logvar*. Recall that when we run a forward inference pass of the model, we output the mean latent representation, mu, and the log-variance, logvar. The logvar can tell us about the model's certainty in the prediction across the embedding dimension for each gene. This uncertainty output is a key feature of BioJEPA-AC that we have not yet seen in the space.

Our eval suite on uncertainty first runs a series of sample-level evaluations and then groups our predictions by perturbation for the remaining correlation assessment. We perform these on the full test set and provide the results broken down by dataset. As you walk through the notebook, you'll see that we evaluate on multiple dimensions to ensure we have a thorough understanding of where our model is working well and where it's struggling.

In [1]:
import numpy as np
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

In [2]:
SEED = 1337
np.random.seed(SEED)

## Data Prep
We'll start by preparing our data. For this evaluation, we need to have our change in expression, or *expression delta*, and our predicted logvar by sample. To get our expression delta, we need the true cell expressions, the control expression for each cell, and the perturbation information. We'll mock up 6 samples across 4 perturbations. A "perturbation" is a unique combination of a sequence, target, modality, and mode applied to a cell type. A perturbation can span datasets. For the 6 samples, we'll show the real control cell expression, predicted control cell expression (running the teacher encoder and then the linear expression decoder), real case cell expression, predicted case cell expression, the logvar, and the perturbation.

For predicted expression, we use the data created by the [linear expression decoder](explainer_eval_decoders_v1_0.ipynb). As a reminder, the decoder takes the mean ($\mu$) BioJEPA-AC output based on the perturbations and cell expression pattern, and then uses a linear layer to project down to a $[\text{n\_genes},\text{1}]$ matrix with a single value per gene representing the expression.

We'll stage data to show a few different predictions: a strong prediction, weak prediction, inverse prediction, and overprediction.

In [3]:
unique_perts = 4
num_genes = 8
num_cells = 6
embed_dim = 4

**Perturbations**

We'll first start with our perturbations. Even though we have 6 cells, our samples will only have 4 unique perturbations. To define a unique perturbation, it's not just about what we target, but the context of it. Because of this, we represent a unique perturbation as $\text{(seq id, targ id, modality id, mode id, cell type)}$. IDs are used since our model keeps the perturbation information in separate caches from our sample expression counts to avoid heavy duplication of information.

We'll also create a mapping of the sample to the perturbation, where the first two samples map to the first perturbation, the next two samples to the second perturbation, and then the final two samples each have a unique perturbation.

In [4]:
pert_keys = [
    (0, 0, 0, 0, 0),  # pert 0: DNA CRISPRi, cell type 0
    (1, 1, 0, 0, 0),  # pert 1: DNA CRISPRi, cell type 0
    (2, 2, 0, 1, 0),  # pert 2: DNA CRISPRa, cell type 0
    (3, 3, 2, 4, 0),  # pert 3: chemical inhibitor, cell type 0
]

sample_to_pert = [0, 0, 1, 1, 2, 3]

**Control Cells**

Next we'll show the control cell values. Recall that for our inference, we pair together a perturbed cell with a random control cell from the same batch. This gives us an approximate change in expression. Beyond just the control cell expression, to calculate our predicted change in expression, we run the linear expression decoder on the control cell's latent representation `z_context`. We'll discuss the calculation more, but to show this we'll also have the predicted control expression.

For the predicted control expression, we'll show a minor shift to highlight that often the prediction is not perfect.

In [5]:
real_control = np.array([
    [2.1, 3.4, 1.2, 4.1, 2.6, 3.1, 1.4, 4.6],
    [1.9, 3.6, 0.8, 3.9, 2.4, 2.9, 1.6, 4.4],
    [2.0, 3.3, 1.1, 4.2, 2.3, 3.2, 1.3, 4.3],
    [2.2, 3.7, 0.9, 3.8, 2.7, 2.8, 1.7, 4.7],
    [2.0, 3.5, 1.0, 4.0, 2.5, 3.0, 1.5, 4.5],
    [2.0, 3.5, 1.0, 4.0, 2.5, 3.0, 1.5, 4.5],
])
pred_control = np.array([
    [2.0, 3.3, 1.3, 4.0, 2.5, 3.2, 1.3, 4.5],
    [1.8, 3.5, 0.9, 3.8, 2.3, 3.0, 1.5, 4.3],
    [1.9, 3.2, 1.2, 4.1, 2.2, 3.3, 1.2, 4.2],
    [2.1, 3.6, 1.0, 3.7, 2.6, 2.9, 1.6, 4.6],
    [1.9, 3.4, 1.1, 3.9, 2.4, 3.1, 1.4, 4.4],
    [1.9, 3.4, 1.1, 3.9, 2.4, 3.1, 1.4, 4.4],
])
real_control.shape, pred_control.shape

((6, 8), (6, 8))

**Case Cells**

Next we'll show the case cell values. In our raw data, we have the real expression of the perturbed cell. We pair this together with the linear expression decoder output based on the mean prediction, mu, from the ACPredictor output. The mean prediction is based on the control cell latent representation `z_context` and the perturbations. You'll quickly be able to see that there is a difference between the real values and the predicted values. We've staged the data so that the first two samples predict closely, the next two samples weakly, the fifth sample predicts in the wrong direction, and the final sample overpredicts. You'll see how these calculations flow through.

In [6]:
real_case = np.array([
    [2.8, 2.3, 1.6, 6.0, 2.0, 4.7, 1.2, 2.9],
    [2.8, 2.2, 1.0, 6.0, 2.0, 4.3, 1.6, 2.5],
    [2.1, 3.2, 1.2, 4.1, 2.4, 3.3, 1.3, 4.4],
    [2.3, 3.6, 0.9, 3.7, 2.7, 2.8, 1.7, 4.7],
    [2.5, 2.7, 1.6, 3.0, 2.8, 3.7, 1.1, 5.7],
    [2.3, 3.1, 1.2, 4.5, 2.2, 3.6, 1.4, 4.9],
])

pred_case = np.array([
    [2.6, 2.3, 1.8, 5.8, 2.0, 4.6, 1.0, 3.0],  # strong
    [2.6, 2.3, 1.2, 5.8, 2.0, 4.5, 1.4, 2.6],  # strong
    [2.05, 3.05, 1.25, 4.05, 2.3, 3.35, 1.15, 4.3],  # weak
    [2.15, 3.6, 0.95, 3.6, 2.65, 2.95, 1.6, 4.65],  # weak
    [1.6, 3.9, 1.6, 4.5, 2.2, 3.9, 1.7, 3.9],  # inverse
    [2.8, 2.2, 1.7, 5.4, 1.5, 4.9, 1.1, 5.6],  # over
])

real_case.shape, pred_case.shape

((6, 8), (6, 8))

**Predicted Sample Logvar**

A key component of our uncertainty calibration is evaluating the log-variance that our model predicts. The logvar is one of the outputs from the ACPredictor based on the input control latent and the perturbation latent. The logvar dimensions match the cell state latents as $[\text{batch},\text{n\_genes},\text{embed\_dim}]$. Since we'll be analyzing this against the actual expression delta, we need to collapse the embedding dimension down to a single value per gene. We'll do this by taking the mean. This will give us a single log-variance value per gene that we can use as our uncertainty. Recall that this value is still in log-variance space. When we want to compare predicted variance against real variance later, we'll exponentiate it back into variance space.

You'll notice that we seed the data with offsets $[-0.2, -0.1, 0.1, 0.2]$ around each target mean to make our calculations traceable. The real data is nowhere near as clean. We've also set up the log-variance values to show high confidence (negatives) and low confidence (positives). As a reminder, this is log-variance, so lower values mean the model is more confident in its prediction.

In [7]:
pred_logvar = np.array([
    [[-2.2, -2.1, -1.9, -1.8], [-2.0, -1.9, -1.7, -1.6], [-2.1, -2.0, -1.8, -1.7], [-1.9, -1.8, -1.6, -1.5], [-2.3, -2.2, -2.0, -1.9], [-1.8, -1.7, -1.5, -1.4], [-2.2, -2.1, -1.9, -1.8], [-1.7, -1.6, -1.4, -1.3]],
    [[0.0, 0.1, 0.3, 0.4], [0.1, 0.2, 0.4, 0.5], [-0.1, 0.0, 0.2, 0.3], [0.2, 0.3, 0.5, 0.6], [-0.2, -0.1, 0.1, 0.2], [0.1, 0.2, 0.4, 0.5], [-0.3, -0.2, 0.0, 0.1], [0.3, 0.4, 0.6, 0.7]],
    [[0.1, 0.2, 0.4, 0.5], [0.2, 0.3, 0.5, 0.6], [-0.1, 0.0, 0.2, 0.3], [0.4, 0.5, 0.7, 0.8], [0.0, 0.1, 0.3, 0.4], [0.3, 0.4, 0.6, 0.7], [-0.2, -0.1, 0.1, 0.2], [0.4, 0.5, 0.7, 0.8]],
    [[-2.1, -2.0, -1.8, -1.7], [-1.9, -1.8, -1.6, -1.5], [-2.0, -1.9, -1.7, -1.6], [-2.2, -2.1, -1.9, -1.8], [-2.0, -1.9, -1.7, -1.6], [-2.1, -2.0, -1.8, -1.7], [-2.0, -1.9, -1.7, -1.6], [-1.9, -1.8, -1.6, -1.5]],
    [[-2.7, -2.6, -2.4, -2.3], [-2.5, -2.4, -2.2, -2.1], [-2.2, -2.1, -1.9, -1.8], [-2.8, -2.7, -2.5, -2.4], [-2.4, -2.3, -2.1, -2.0], [-2.3, -2.2, -2.0, -1.9], [-2.6, -2.5, -2.3, -2.2], [-2.9, -2.8, -2.6, -2.5]],
    [[0.0, 0.1, 0.3, 0.4], [0.3, 0.4, 0.6, 0.7], [-0.3, -0.2, 0.0, 0.1], [0.6, 0.7, 0.9, 1.0], [0.1, 0.2, 0.4, 0.5], [0.4, 0.5, 0.7, 0.8], [-0.4, -0.3, -0.1, 0.0], [0.2, 0.3, 0.5, 0.6]],
])
pred_logvar.shape, pred_logvar

((6, 8, 4),
 array([[[-2.2, -2.1, -1.9, -1.8],
         [-2. , -1.9, -1.7, -1.6],
         [-2.1, -2. , -1.8, -1.7],
         [-1.9, -1.8, -1.6, -1.5],
         [-2.3, -2.2, -2. , -1.9],
         [-1.8, -1.7, -1.5, -1.4],
         [-2.2, -2.1, -1.9, -1.8],
         [-1.7, -1.6, -1.4, -1.3]],
 
        [[ 0. ,  0.1,  0.3,  0.4],
         [ 0.1,  0.2,  0.4,  0.5],
         [-0.1,  0. ,  0.2,  0.3],
         [ 0.2,  0.3,  0.5,  0.6],
         [-0.2, -0.1,  0.1,  0.2],
         [ 0.1,  0.2,  0.4,  0.5],
         [-0.3, -0.2,  0. ,  0.1],
         [ 0.3,  0.4,  0.6,  0.7]],
 
        [[ 0.1,  0.2,  0.4,  0.5],
         [ 0.2,  0.3,  0.5,  0.6],
         [-0.1,  0. ,  0.2,  0.3],
         [ 0.4,  0.5,  0.7,  0.8],
         [ 0. ,  0.1,  0.3,  0.4],
         [ 0.3,  0.4,  0.6,  0.7],
         [-0.2, -0.1,  0.1,  0.2],
         [ 0.4,  0.5,  0.7,  0.8]],
 
        [[-2.1, -2. , -1.8, -1.7],
         [-1.9, -1.8, -1.6, -1.5],
         [-2. , -1.9, -1.7, -1.6],
         [-2.2, -2.1, -1.9, -1.8],

In [8]:
sample_logvar = pred_logvar.mean(axis=-1)
sample_logvar.shape, sample_logvar

((6, 8),
 array([[-2.0000000e+00, -1.8000000e+00, -1.9000000e+00, -1.7000000e+00,
         -2.1000000e+00, -1.6000000e+00, -2.0000000e+00, -1.5000000e+00],
        [ 2.0000000e-01,  3.0000000e-01,  1.0000000e-01,  4.0000000e-01,
         -6.9388939e-18,  3.0000000e-01, -1.0000000e-01,  5.0000000e-01],
        [ 3.0000000e-01,  4.0000000e-01,  1.0000000e-01,  6.0000000e-01,
          2.0000000e-01,  5.0000000e-01, -6.9388939e-18,  6.0000000e-01],
        [-1.9000000e+00, -1.7000000e+00, -1.8000000e+00, -2.0000000e+00,
         -1.8000000e+00, -1.9000000e+00, -1.8000000e+00, -1.7000000e+00],
        [-2.5000000e+00, -2.3000000e+00, -2.0000000e+00, -2.6000000e+00,
         -2.2000000e+00, -2.1000000e+00, -2.4000000e+00, -2.7000000e+00],
        [ 2.0000000e-01,  5.0000000e-01, -1.0000000e-01,  8.0000000e-01,
          3.0000000e-01,  6.0000000e-01, -2.0000000e-01,  4.0000000e-01]]))

**Calculate Sample Delta**

A major component of our expression benchmark is not looking at absolute predictions, but the change in expression. Some claim that this simplifies the task. Biologically, we see this as addressing the important questions: can you predict what will change, in what direction, and by how much? We focus on calculating two sample-level differences:

1. `pred_delta` - the predicted change in expression as calculated by $\hat{\delta}_g = \hat{x}^{\text{case}}_g - \hat{x}^{\text{ctrl}}_g$. This value compares the predicted perturbed expression (`pred_case`) against the predicted control expression (`pred_control`). We use the predicted control expression to isolate BioJEPA-AC's learned perturbation effect from any baseline reconstruction error.
2. `real_delta` - the real change in expression as calculated by $\delta_g = x^{\text{case}}_g - x^{\text{ctrl}}_g$. This is our source of truth.

With this calculation, you'll see how we get both increases and decreases in expression by gene. We'll end up comparing these by different slices in our calculations.

In [9]:
pred_delta = pred_case - pred_control

pred_delta.shape, pred_delta

((6, 8),
 array([[ 0.6 , -1.  ,  0.5 ,  1.8 , -0.5 ,  1.4 , -0.3 , -1.5 ],
        [ 0.8 , -1.2 ,  0.3 ,  2.  , -0.3 ,  1.5 , -0.1 , -1.7 ],
        [ 0.15, -0.15,  0.05, -0.05,  0.1 ,  0.05, -0.05,  0.1 ],
        [ 0.05,  0.  , -0.05, -0.1 ,  0.05,  0.05,  0.  ,  0.05],
        [-0.3 ,  0.5 ,  0.5 ,  0.6 , -0.2 ,  0.8 ,  0.3 , -0.5 ],
        [ 0.9 , -1.2 ,  0.6 ,  1.5 , -0.9 ,  1.8 , -0.3 ,  1.2 ]]))

In [10]:
real_delta = real_case - real_control

real_delta.shape, real_delta

((6, 8),
 array([[ 0.7, -1.1,  0.4,  1.9, -0.6,  1.6, -0.2, -1.7],
        [ 0.9, -1.4,  0.2,  2.1, -0.4,  1.4,  0. , -1.9],
        [ 0.1, -0.1,  0.1, -0.1,  0.1,  0.1,  0. ,  0.1],
        [ 0.1, -0.1,  0. , -0.1,  0. ,  0. ,  0. ,  0. ],
        [ 0.5, -0.8,  0.6, -1. ,  0.3,  0.7, -0.4,  1.2],
        [ 0.3, -0.4,  0.2,  0.5, -0.3,  0.6, -0.1,  0.4]]))

## Sample Level

Our first set of uncertainty evaluations all happen at the sample level. We're trying to understand whether the model's predicted uncertainty aligns with its actual prediction error. We'll compare uncertainty to our prediction error, then analyze our uncertainty by binning it, and finally analyze normalized uncertainty.

**Uncertainty**

We first need to convert our logvar into a single value per sample that we'll call *uncertainty*. We'll run a simple mean on the logvar using the calculation:
$$
\text{uncertainty}_\text{sample} = \frac{1}{G}\sum_{g=1}^{G} \text{logvar}_{g}
$$

**Mean Squared Error**

Now we need to know what to compare it against. The logvar should show the variance within which our model prediction is correct. To analyze this, we'll look at the mean squared error (MSE) of our predicted expression delta against the real expression delta. We calculate this as:
$$
\text{MSE}_\text{sample} = \frac{1}{G}\sum_{g=1}^{G} (\hat{\delta}_{g} - \delta_{g})^2
$$

Since all 8 genes in our staged data are measured, we'll calculate across every gene. In our production evaluation, different datasets can measure different genes, so we only calculate across the genes measured for each sample.

Our comparisons will then look to see how well our MSE aligns with our uncertainty. We'll start by calculating the sample uncertainty and MSE. You can see in the uncertainty which samples have high confidence (low uncertainty) and which don't. Similarly, you can also see that while we have some samples with high uncertainty, their MSE is low, showing they're uncertain but correct.

In [11]:
sample_unc = sample_logvar.mean(axis=1)
sample_unc.shape, sample_unc

((6,), array([-1.825 ,  0.2125,  0.3375, -1.825 , -2.35  ,  0.3125]))

In [12]:
sample_mse = np.mean((pred_delta - real_delta)**2, axis=1)
sample_mse.shape, sample_mse

((6,),
 array([0.0175   , 0.0175   , 0.001875 , 0.0028125, 1.0675   , 0.58     ]))

### Pearson Uncertainty Correlation

Now that we know our error and uncertainty, we're ready to start evaluating how they are correlated. By analyzing the correlation between MSE and uncertainty, we can learn whether the model knows when it's wrong. The more positively correlated they are, the better our uncertainty prediction is and the more useful it is for understanding our model's confidence. Pearson correlation measures how linearly correlated the uncertainty and MSE are across all samples, regardless of scale. We calculate it as:
$$
r_{\text{unc}} = \frac{\sum_{s=1}^{S}(u_s - \bar{u})(\text{MSE}_s - \overline{\text{MSE}})}{\sqrt{\sum_{s=1}^{S}(u_s - \bar{u})^2} \cdot \sqrt{\sum_{s=1}^{S}(\text{MSE}_s - \overline{\text{MSE}})^2}}
$$
Each sample contributes one uncertainty value and one MSE value, and we correlate across all $S$ samples. A more positive correlation means the model better understands when it's correct (both low uncertainty when correct and high uncertainty when wrong). Because of how we staged the data, we'll see an anti-correlation here (negative), which means the model is most confident on the samples where it makes the biggest mistakes, and least confident where it's actually accurate.

*A strong negative correlation is still informative because it tells us that the model's uncertainty is moving in the opposite direction of its error. It does not mean the uncertainty is useful as-is, since the model is most confident on its largest errors. A correlation near zero is different: it tells us there is no consistent relationship between uncertainty and error.*

In [13]:
pearson_r, _ = pearsonr(sample_unc, sample_mse)
pearson_r = float(pearson_r)
pearson_r

-0.32247488236996175

### Spearman Uncertainty Correlation

With Spearman's rank correlation, we can calculate the rank order correlation of our uncertainty. We calculate it as:
$$
\rho_{\text{unc}} = 1 - \frac{6\sum_{s=1}^{S}(R(u_s) - R(\text{MSE}_s))^{2}}{S(S^{2}-1)}
$$

The rank is the position of each value when sorted from smallest to largest. With Spearman's correlation, we can determine whether the predicted uncertainty and MSE have the same ordering. Similar to Pearson, a positive correlation means the uncertainty and MSE have the same ordering, while a negative correlation means the ordering is reversed. Again, because of how we staged our data, we'll see a fairly strong anti-correlation.

In [14]:
spearman_r, _ = spearmanr(sample_unc, sample_mse)
spearman_r = float(spearman_r)
spearman_r

-0.6

## Bin-Based Sample Analysis
Our next set of analyses looks at our uncertainty from a bin, or grouped, perspective. For these analyses, we create a certain number of groupings to equally distribute our samples based on their uncertainty. We then review the mean error of each bin, calculated as:
$$
\bar{e}_k = \frac{1}{|B_k|}\sum_{s \in B_k} \text{MSE}_s
$$
Where $B_k$ is the set of samples whose uncertainty $u_s$ falls in the $k$-th percentile bin. By binning our uncertainty, we can understand if there are certain groupings of our uncertainty that are better than others. For example, if performance is high in our low uncertainty bins and drops as uncertainty increases, our uncertainty is useful. If we see the opposite, that tells us our uncertainty is moving in the wrong direction.

We'll start by creating our bins and then calculating the mean error within each bin.

In [15]:
n_bins = 3
bin_mean_error = []

**Create Bins**

Since we only have 6 samples, we'll create 3 bins to distribute the samples across. This will pull 2 samples into each bin (though in our real data, our bins won't have the exact same number of samples). We'll use percentages to create the bins and then compute the bin edges from our sample uncertainty.

*Note that we create one more edge than the number of bins since we need the high and low of each bin.*

In [16]:
bin_perc = np.linspace(0, 100, n_bins + 1)
bin_edges = np.percentile(sample_unc, bin_perc)
bin_perc, bin_edges

(array([  0.        ,  33.33333333,  66.66666667, 100.        ]),
 array([-2.35      , -1.825     ,  0.24583333,  0.3375    ]))

### Bin Analysis

Now that we have our bins, we'll calculate our bin analysis, which is the average error by bin. Recall that our bin edges are based on uncertainty, but our bin error is based on MSE. To calculate our bin mean error, we'll use the bin edge values and compare them with our sample uncertainty to identify which samples belong in each bin. When we have those IDs, we can then pull out the sample MSE and take the mean to get a single bin mean error.

Because of how we set up our sample data, you'll see that we have very high error in our first bin, dominated by a single sample, while our middle bin will have the lowest.

In [17]:
for i in range(n_bins):
    print(f'---- Bin {i} ----')
    mask = (sample_unc >= bin_edges[i]) & (sample_unc <= bin_edges[i + 1] if i == n_bins - 1 else sample_unc < bin_edges[i + 1])
    print(mask)
    print(sample_mse[mask])
    bin_mean_error.append(sample_mse[mask].mean())
bin_mean_error

---- Bin 0 ----
[ True False False False  True False]
[0.0175 1.0675]
---- Bin 1 ----
[False  True False  True False False]
[0.0175    0.0028125]
---- Bin 2 ----
[False False  True False False  True]
[0.001875 0.58    ]


[np.float64(0.5425000000000001),
 np.float64(0.010156250000000016),
 np.float64(0.2909374999999999)]

In [18]:
bin_analysis = {'n_bins': n_bins, 'bin_mean_errors': [float(e) for e in bin_mean_error]}
bin_analysis

{'n_bins': 3,
 'bin_mean_errors': [0.5425000000000001,
  0.010156250000000016,
  0.2909374999999999]}

### Monotonicity Score

If you think about it, since our bins are ordered by increasing uncertainty, we would expect that MSE increases as the uncertainty increases (directly correlated). We now need to figure out if that's true. For this, we calculate the monotonicity score. This score is the fraction of consecutive bins where error increases as uncertainty increases. We calculate it as:
$$
\text{Monotonicity}_\text{sample\_bins} = \frac{1}{K-1}\sum_{k=1}^{K-1} \mathbf{1}[\bar{e}_{k+1} > \bar{e}_{k}]
$$

A score of 1.0 means error rises monotonically with uncertainty (perfect calibration), while a low score means the error-uncertainty relationship is noisy or inverted. If you look at the calculated bin data, you can see that our error drops from the first to the second bin, but then rises, meaning half the steps are increases. You'll see we get the expected $0.5$.

In [19]:
monotonicity = sum(1 for i in range(len(bin_mean_error) - 1) if bin_mean_error[i + 1] > bin_mean_error[i]) / (n_bins - 1)
monotonicity

0.5

### Selective Prediction
Our next analysis is selective prediction. To calculate this, we sort samples by uncertainty and compute MSE for the top 25/50/75/100% most confident predictions, based on the uncertainty. We calculate it as:
$$
\text{MSE}_{\text{percentile}} = \frac{1}{|S_p|}\sum_{s \in S_p} \text{MSE}_s
$$
Where $S_p$ is the $p\%$ of samples with the lowest $u_s$ (most confident). For a well-trained model, we would expect MSE to rise as we include less confident samples. Selective prediction looks to validate the theory that if the uncertainty is meaningful, we should be able to improve our model's accuracy by only trusting its most confident predictions. What we'll actually see is that because we structured our data to have high confidence in bad predictions and low confidence in accurate predictions, we get the inverse: our error drops as we increase the percentage.

In [20]:
selective = {}
sort_idx = np.argsort(sample_unc)
sort_idx, sample_mse[sort_idx]

(array([4, 0, 3, 1, 5, 2]),
 array([1.0675   , 0.0175   , 0.0028125, 0.0175   , 0.58     , 0.001875 ]))

In [21]:
for pct in [25, 50, 75, 100]:
    n = max(1, int(len(sort_idx) * pct / 100))
    idx = sort_idx[:n]
    selective[f'top_{pct}pct'] = {'mse': float(np.mean(sample_mse[idx])), 'n_samples': n}
selective

{'top_25pct': {'mse': 1.0675000000000001, 'n_samples': 1},
 'top_50pct': {'mse': 0.3626041666666668, 'n_samples': 3},
 'top_75pct': {'mse': 0.2763281250000001, 'n_samples': 4},
 'top_100pct': {'mse': 0.2811979166666667, 'n_samples': 6}}

### Expected Calibration Error (ECE)

Up to now, our bin analysis has assumed a direct relationship: the higher the uncertainty, the higher we expect the error. ECE first normalizes uncertainty and MSE so that we can compare them on the same scale, then measures the average gap between them across bins. We calculate it as:

$$
\text{ECE} = \sum_{k=1}^{K} \frac{|B_k|}{S} \left| \bar{u}_k - \bar{e}_k \right|
$$

Where $\bar{u}_k$ and $\bar{e}_k$ are the mean normalized uncertainty and mean normalized error within bin $k$. A perfectly calibrated model has an ECE of 0, meaning its uncertainty level directly matches the error level in every bin.

In [22]:
ece = 0.0

**Normalize Uncertainty and MSE**

We'll first normalize our uncertainty and error. You'll see that in both cases, because of the normalization, our values end up between $[0, 1]$.

In [23]:
unc_norm = (sample_unc - sample_unc.min()) / (sample_unc.max() - sample_unc.min() + 1e-8)
unc_norm

array([0.19534884, 0.95348837, 1.        , 0.19534884, 0.        ,
       0.99069767])

In [24]:
err_norm = (sample_mse - sample_mse.min()) / (sample_mse.max() - sample_mse.min() + 1e-8)
err_norm

array([1.46627565e-02, 1.46627565e-02, 0.00000000e+00, 8.79765388e-04,
       9.99999991e-01, 5.42521989e-01])

**Binned Calculation**

While we're reusing the bin count, we do need to change how the bins are created. Our earlier analysis uses percentile bins based on raw uncertainty, while ECE creates equal-width bins from 0 to 1 based on normalized uncertainty. If you look at our normalized uncertainty, we have a bimodal distribution where a few values are very small and a few are very high. Because of this, you'll see that our middle bin has no samples. Our ECE is around 0.5, meaning the uncertainty and error are, on average, off by about half of the normalized scale and are poorly calibrated.

In [25]:
for i in range(n_bins):
    print(f'---- Bin {i} ----')
    low = i / n_bins
    high = (i + 1) / n_bins if i < n_bins - 1 else 1.0
    mask = (unc_norm >= low) & (unc_norm < high if i < n_bins - 1 else unc_norm <= high)
    print(f'Bin from {low} to {high}')
    if mask.sum() == 0:
        print('no samples in bin')
        continue
    print(mask)
    bin_weight = mask.sum() / len(sample_mse)
    bin_unc_mean = unc_norm[mask].mean()
    bin_err_mean = err_norm[mask].mean()
    bin_ece = bin_weight * abs(bin_unc_mean - bin_err_mean)
    print(f'Bin values: weight {bin_weight} | Uncertainty mean {bin_unc_mean} | MSE mean {bin_err_mean} | ECE: {bin_ece}')
    ece += bin_ece

ece

---- Bin 0 ----
Bin from 0.0 to 0.3333333333333333
[ True False False  True  True False]
Bin values: weight 0.5 | Uncertainty mean 0.13023255765494865 | MSE mean 0.3385141708213724 | ECE: 0.10414080658321188
---- Bin 1 ----
Bin from 0.3333333333333333 to 0.6666666666666666
no samples in bin
---- Bin 2 ----
Bin from 0.6666666666666666 to 1.0
[False  True  True False False  True]
Bin values: weight 0.5 | Uncertainty mean 0.9813953451855056 | MSE mean 0.18572824850147485 | ECE: 0.3978335483420154


np.float64(0.5019743549252272)

## Perturbation Level

Our next set of evals is done at the perturbation level. As a reminder, we consider a perturbation to be unique if it's the same sequence, target, modality, and mode applied to the same cell type. To analyze per perturbation, we review all of the sample-level uncertainty we've calculated and group it by perturbation. For each unique perturbation, we calculate the mean.

Since all our staged samples have one perturbation, we'll use all of them here. In our production evaluation, we remove samples with more than one perturbation before calculating our perturbation-level correlations and variance. Those samples are still included in the sample-level evaluations above.

Once we have the perturbation-level means, we'll again calculate a few different correlations between error and uncertainty.

**Per-Perturbation Mean**

Our first calculations will be per-perturbation mean uncertainty and MSE. We take the mean using:
$$\begin{aligned}
\bar{u}_p &= \frac{1}{|R_p|}\sum_{s \in R_p} u_s  \\\\
\overline{\text{MSE}}_p &= \frac{1}{|R_p|}\sum_{s \in R_p} \text{MSE}_s
\end{aligned}$$
Where $R_p$ is the set of replicate samples for perturbation $p$. Because of how we staged our data, you can see more clearly the uncertainty and error not aligning on our third perturbation.

In [26]:
pert_unc = np.array([np.mean(sample_unc[np.array(sample_to_pert) == p]) for p in range(unique_perts)])
pert_unc

array([-0.80625, -0.74375, -2.35   ,  0.3125 ])

In [27]:
pert_mse = np.array([np.mean(sample_mse[np.array(sample_to_pert) == p]) for p in range(unique_perts)])
pert_mse

array([0.0175    , 0.00234375, 1.0675    , 0.58      ])

### Pearson Uncertainty Correlation Per Perturbation

Now that we know our per-perturbation error and uncertainty, we're ready to start evaluating how they are correlated. As before, we'll evaluate the linear correlation between the uncertainty and MSE by calculating Pearson's correlation. We calculate it as:
$$
r_{\text{pert}} = \frac{\sum_{p=1}^{P}(a_p - \bar{a})(b_p - \bar{b})}{\sqrt{\sum_{p=1}^{P}(a_p - \bar{a})^2} \cdot \sqrt{\sum_{p=1}^{P}(b_p - \bar{b})^2}}
$$
Where $a_p = \bar{u}_p$ and $b_p = \overline{\text{MSE}}_p$ are the per-perturbation means. You'll see that even though we have some good correlation for 3 of the 4 perturbations, the size of the difference in the third perturbation impacts our correlation heavily.

In [28]:
pert_pearson_r, _ = pearsonr(pert_unc, pert_mse)
pert_pearson_r = float(pert_pearson_r)
pert_pearson_r

-0.5051361586230405

### Spearman Uncertainty Correlation Per Perturbation

We'll also now calculate the rank order correlation, again using Spearman. We calculate it as:

$$
\rho_{\text{pert}} = 1 - \frac{6\sum_{p=1}^{P}(R(\bar{u}_p) - R(\overline{\text{MSE}}_p))^{2}}{P(P^{2}-1)}
$$

Because of the error on our third perturbation, we'll again see that it pulls our correlation negative.

In [29]:
pert_spearman_r, _ = spearmanr(pert_unc, pert_mse)
pert_spearman_r = float(pert_spearman_r)
pert_spearman_r

-0.39999999999999997

### Variance Coefficient of Determination $R^2$

Since we're predicting log-variance (logvar), we also want to evaluate whether the predicted variance matches the observed variance across replicates. For this, we exponentiate the logvar back to variance space, average across replicates per perturbation, and calculate the coefficient of determination against the observed variance of the real expression delta. We calculate this as:
$$\begin{aligned}
\hat{\sigma}^2_{p,g} &= \frac{1}{|R_p|}\sum_{s \in R_p} e^{\text{logvar}_{s,g}} \\
\sigma^2_{p,g} &= \text{Var}_{s \in R_p}(\delta_{s,g}) \\
R^2_{\text{var}} &= \frac{1}{P}\sum_{p=1}^{P} R^2(\sigma^2_{p}, \hat{\sigma}^2_{p})
\end{aligned}$$

For each perturbation, we use the different observed expression deltas to calculate a variance. Because of this, in the production eval, we only evaluate perturbations with 3 or more samples, but for this example we'll do it for 2 or more (if we only did 1, the variance would be 0). You'll also see that with such a small sample count, even tiny errors quickly make this eval plummet to extremely bad values. A perfect score would be $+1.0$.

In [30]:
variance_r2s = []

In [31]:
for p in range(unique_perts):
    print(f'---- Pert {p} ----')
    idx = np.array(sample_to_pert) == p
    if sum(idx) < 2:
        print('not enough samples')
        continue
    print(idx)
    real_var = np.var(real_delta[idx], axis=0)
    pred_var = np.mean(np.exp(sample_logvar[idx]), axis=0)
    pert_r2 = float(r2_score(real_var, pred_var))
    print(real_var)
    print(pred_var)
    print(f'R^2 {pert_r2}')
    variance_r2s.append(pert_r2)

---- Pert 0 ----
[ True  True False False False False]
[0.01   0.0225 0.01   0.01   0.01   0.01   0.01   0.01  ]
[0.67836902 0.75757885 0.62736977 0.83725411 0.56122821 0.77587766
 0.52008635 0.93592572]
R^2 -29698.66087025179
---- Pert 1 ----
[False False  True  True False False]
[4.93038066e-32 4.93038066e-32 2.50000000e-03 1.97215226e-31
 2.50000000e-03 2.50000000e-03 0.00000000e+00 2.50000000e-03]
[0.74971371 0.83725411 0.6352349  0.97872704 0.69335082 0.89914494
 0.58264944 1.00240116]
R^2 -419347.39235135104
---- Pert 2 ----
not enough samples
---- Pert 3 ----
not enough samples


In [32]:
r2_mean = float(np.mean(variance_r2s))
r2_mean

-224523.0266108014

## Uncertainty Calibration Final Wrap-Up
We've now walked through our evaluation of our predicted uncertainty, logvar. As you can see, we review the uncertainty in a number of different ways, comparing it against both prediction error and the real variance we observe in the data. Our walkthrough shows the evaluations as we run them across all our test data, but our evaluation suite also calculates these values broken down by dataset. Given that uncertainty prediction is unique to BioJEPA-AC in this class of models, these evaluations are a first step in understanding how to better use this type of data.